# البرنامج التعليمي لشجرة العائلة باستخدام UnifyWeaver

يوضح هذا الدفتر التفاعلي للملاحظات كيفية استخدام UnifyWeaver لتجميع محددات Prolog إلى سكربتات Bash.

## المتطلبات الأساسية

- تثبيت SWI-Prolog
- توفر مكتبة UnifyWeaver
- تثبيت نواة Jupyter الخاصة بـ Prolog (`pip install prolog-jupyter-kernel`)

## الأهداف التعليمية

بنهاية هذا الدفتر، ستكون قادرًا على:
1. تعريف حقائق وقواعد Prolog
2. استخدام UnifyWeaver لتجميع المحددات إلى Bash
3. اختبار سكربتات Bash المولدة
4. فهم تجميع الانغلاق المتعدي

## الخطوة 1: تهيئة بيئة UnifyWeaver

أولاً، نحتاج إلى تحميل وحدات UnifyWeaver. سنستخدم ملف `init.pl` من دليل education.

In [ ]:
% تحميل ملف التهيئة
['../init'].

## الخطوة 2: تعريف علاقات العائلة

دعنا نحدد بعض علاقات الآباء والأبناء (parent-child) من شجرة العائلة المذكورة في السياق التاريخي.

In [ ]:
% تعريف حقائق parent
:- dynamic parent/2.

parent(abraham, isaac).
parent(abraham, ishmael).
parent(isaac, jacob).
parent(isaac, esau).
parent(jacob, reuben).
parent(jacob, simeon).
parent(jacob, levi).
parent(jacob, judah).

## الخطوة 3: اختبار استعلامات الوالدين (Parent)

قبل التجميع، دعنا نتحقق من صحة بياناتنا باستخدام بعض استعلامات Prolog.

In [ ]:
% استعلام: من هم أبناء إبراهيم؟
parent(abraham, Child).

In [ ]:
% استعلام: من هم أبناء يعقوب؟
parent(jacob, Child).

## الخطوة 4: تعريف علاقة السلف (Ancestor)

الآن دعنا نحدد الانغلاق المتعدي — علاقة `ancestor`.

In [ ]:
% تعريف ancestor على أنه الانغلاق المتعدي لـ parent
:- dynamic ancestor/2.

% الحالة الأساسية: الوالد هو سلف
ancestor(X, Y) :- parent(X, Y).

% الحالة العودية: إذا كان X والدًا لـ Y وكان Y سلفًا لـ Z، فإن X هو سلف لـ Z
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

## الخطوة 5: اختبار استعلامات السلف (Ancestor)

دعنا نتحقق من أن محدد ancestor يعمل بشكل صحيح.

In [ ]:
% استعلام: هل إبراهيم سلف ليعقوب؟
( ancestor(abraham, jacob) ->
    writeln('Yes: Abraham is an ancestor of Jacob')
;
    writeln('No: Abraham is not an ancestor of Jacob')
).

In [ ]:
% استعلام: من هم جميع نسل إبراهيم؟
ancestor(abraham, Descendant).

## الخطوة 6: تجميع Parent إلى Bash

الآن إلى الجزء الممتع — دعنا نجمع حقائق `parent/2` إلى سكربت Bash!

In [ ]:
% تحميل مترجم التدفق
\+ \+ (
    use_module(unifyweaver(core/stream_compiler)),

    % تجميع حقائق parent إلى bash
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    writeln('Generated Bash code for parent/2:'),
    writeln(_BashCode)
).

## الخطوة 7: حفظ سكربت Parent

دعنا نحفظ كود Bash المولد في ملف.

In [ ]:
% حفظ في ملف
\+ \+ (
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    setup_call_cleanup(
        open('../output/parent.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/parent.sh')
).

## الخطوة 8: تجميع Ancestor إلى Bash

الآن دعنا نجمع محدد `ancestor/2`، الذي يستخدم العودية.

In [ ]:
% تحميل المترجم العودي
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),

    % تجميع ancestor إلى bash
    compile_recursive(ancestor/2, [], _BashCode),
    writeln('Generated Bash code for ancestor/2:'),
    writeln(_BashCode)
).

## الخطوة 9: حفظ سكربت Ancestor

حفظ سكربت ancestor في ملف.

In [ ]:
% حفظ في ملف
\+ \+ (
    compile_recursive(ancestor/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/ancestor.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/ancestor.sh')
).

## الخطوة 10: اختبار السكربتات المولدة

دعنا الآن نختبر سكربتات Bash المولدة! سنستخدم الأمر السحري `%%bash` لتشغيل أوامر bash.

In [ ]:
%%bash
# استيراد سكربت parent باستخدام source
source ../output/parent.sh

# اختبار: من هم أبناء إبراهيم؟
echo "أبناء إبراهيم:"
parent abraham

In [ ]:
%%bash
# استيراد كلا السكربتين باستخدام source
source ../output/parent.sh
source ../output/ancestor.sh

# اختبار: من هم نسل إبراهيم؟
echo "نسل إبراهيم:"
ancestor abraham

In [ ]:
%%bash
# استيراد كلا السكربتين باستخدام source
source ../output/parent.sh
source ../output/ancestor.sh

# اختبار: هل إبراهيم سلف ليهوذا؟
if ancestor abraham judah >/dev/null 2>&1; then
    echo "✓ نعم، إبراهيم سلف ليهوذا"
else
    echo "✗ لا"
fi

## الخطوة 11: فهم استراتيجية التجميع

دعنا نحلل ما قام به UnifyWeaver:

1. **تجميع parent**: استخدام `stream_compiler` لإنشاء دالة تدفق بسيطة تُخرج جميع أزواج الوالد-الطفل

2. **تجميع ancestor**: اكتشاف نمط الانغلاق المتعدي واستخدام تحسين البحث بالاتساع (BFS) لحساب جميع الأسلاف الذين يمكن الوصول إليهم بكفاءة

دعنا نتحقق من استراتيجية التجميع:

In [ ]:
% التحقق مما إذا كان ancestor مصنفًا على أنه عودي
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),
    recursive_compiler:classify_predicate(ancestor/2, _Classification),
    format('Ancestor classification: ~w~n', [_Classification])
).

## الملخص

في هذا الدفتر، تعلمت:

✅ كيفية تعريف حقائق وقواعد Prolog

✅ كيفية استخدام `stream_compiler` في UnifyWeaver للحقائق

✅ كيفية استخدام `recursive_compiler` في UnifyWeaver للمحددات العودية

✅ كيفية اختبار سكربتات Bash المولدة

✅ أن UnifyWeaver يكتشف الانغلاق المتعدي تلقائيًا ويطبق تحسين BFS

## الخطوات التالية

جرب هذه التمارين:

1. أضف المزيد من أفراد العائلة إلى الشجرة
2. عرّف محدد `grandparent/2` (الجد/الجدة) وقم بتجميعه
3. أنشئ محدد `sibling/2` (شخصان لهما نفس الوالد)
4. استكشف كود Bash المولد لفهم خوارزمية BFS

تابع إلى **دفتر الملاحظات 2: مقارنة أنماط العودية** للتعرف على أنماط العودية المتقدمة!